# Predicción de Churn Telefonía móvil

Las comunicaciones por teléfono móvil siempre han sido objeto de una feroz competencia, en la que, usualmente, hay un número reducido de competidores (operadores) en el mercado que oscila entre cuatro y seis. Los operadores deben hacer fuertes inversiones de entrada para poder operar y prestar un servicio adecuado. El mercado de telefonía móvil está usualmente saturado (poca posibilidad de ampliación del mercado) y, por tanto, el conjunto de consumidores disponibles es reducido, por lo que los operadores usualmente cambian su estrategia de adquisición de clientes a una de retención, pues en el mundo se calcula que el costo de adquirir un nuevo cliente es cinco veces superior al de retenerlo- El esfuerzo fundamental se centra en predecir la deserción (churn) para poderla prevenir.

Un operador de telecomunicaciones les ha contratado para tener una mejor predicción de churn. Les otorga una base de datos de entrenamiento (que es una muestra aleatoria simple de la base de datos total de la compañía) que contiene el resultado de retención en los últimos seis meses y les pide que se realice la mejor predicción posible (basada en el área bajo la curva COR) para una base de datos de prueba en un concurso de Kaggle. Pide también un reporte con metodología CRISP-DM (de máximo 6 páginas) de los modelos realizados.

## Importación de librerias y datos

In [32]:
%pip install pandas openpyxl scikit-learn imblearn xgboost

   ---------------------------------------- 0.0/124.9 MB ? eta -:--:--
   ---------------------------------------- 1.3/124.9 MB 8.4 MB/s eta 0:00:15
   - -------------------------------------- 3.1/124.9 MB 8.8 MB/s eta 0:00:14
   - -------------------------------------- 4.7/124.9 MB 8.4 MB/s eta 0:00:15
   -- ------------------------------------- 6.6/124.9 MB 8.4 MB/s eta 0:00:15
   -- ------------------------------------- 8.4/124.9 MB 8.5 MB/s eta 0:00:14
   --- ------------------------------------ 10.0/124.9 MB 8.4 MB/s eta 0:00:14
   --- ------------------------------------ 11.8/124.9 MB 8.5 MB/s eta 0:00:14
   ---- ----------------------------------- 13.1/124.9 MB 8.4 MB/s eta 0:00:14
   ---- ----------------------------------- 14.9/124.9 MB 8.3 MB/s eta 0:00:14
   ----- ---------------------------------- 17.3/124.9 MB 8.6 MB/s eta 0:00:13
   ------ --------------------------------- 19.1/124.9 MB 8.6 MB/s eta 0:00:13
   ------ --------------------------------- 21.0/124.9 MB 8.7 MB/

In [3]:
import pandas as pd
import numpy as np

In [4]:
data_train = pd.read_excel('./files/traintelco.xlsx')
data_test = pd.read_excel('./files/testelco.xlsx')

In [5]:
data_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8243 entries, 0 to 8242
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id                     8243 non-null   int64         
 1   Fecha de nacimiento    8243 non-null   datetime64[ns]
 2   tipo cliente           8243 non-null   int64         
 3   Factura online         8243 non-null   int64         
 4   Antigüedad Equipo      8243 non-null   int64         
 5   Plan de datos          8243 non-null   int64         
 6   facturación            8243 non-null   int64         
 7   mora                   8243 non-null   int64         
 8   Fecha inicio contrato  8243 non-null   datetime64[ns]
 9   minutos                8243 non-null   int64         
 10  resultado              8243 non-null   int64         
dtypes: datetime64[ns](2), int64(9)
memory usage: 708.5 KB


In [6]:
data_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id                     2000 non-null   int64         
 1   Fecha de nacimiento    2000 non-null   datetime64[ns]
 2   tipo cliente           2000 non-null   int64         
 3   Factura online         2000 non-null   int64         
 4   Antigüedad Equipo      2000 non-null   int64         
 5   Plan de datos          2000 non-null   int64         
 6   facturación            2000 non-null   int64         
 7   mora                   2000 non-null   int64         
 8   Fecha inicio contrato  2000 non-null   datetime64[ns]
 9   minutos                2000 non-null   int64         
dtypes: datetime64[ns](2), int64(8)
memory usage: 156.4 KB


## Limpieza de datos

In [7]:
data_train_clean = data_train.copy()
data_test_clean = data_test.copy()

Calculo de la edad de la persona para reemplazarlo por la fecha de nacimiento, se considera como un dato predictivo de mayor peso

In [8]:
# Calculate age from birth year and drop birth year
def calculate_age(df):
    df['age'] = 2018 - df['Fecha de nacimiento'].dt.year
    df = df.drop(columns=['Fecha de nacimiento'])
    return df

data_train_clean = calculate_age(data_train_clean)
data_test_clean = calculate_age(data_test_clean)

Dummificación de el tipo de cliente

In [9]:
# Dummy variables for 'tipo cliente' and drop 'tipo cliente'
def dummy(df, columns):
    df = pd.get_dummies(df, columns=columns)
    df = df.rename(columns={'tipo cliente_1': 'hombre', 'tipo cliente_2': 'mujer', 'tipo cliente_3': 'empresarial'})
    return df

columns = ['tipo cliente']
data_train_clean = dummy(data_train_clean, columns)
data_test_clean = dummy(data_test_clean, columns)

Se cambia el valor de fecha de contrato por una variable cuantitativa que representa qué tan nuevo es el contrato

In [10]:
# Extract timestamp from date
def extract_timestamp(df):
    column = 'Fecha inicio contrato'
    df[column] = pd.to_datetime(df[column])
    df[column] = df[column].astype(np.int64) // 10**9
    df = df.rename(columns={'Fecha inicio contrato': 'novedad del contrato'})
    return df

data_train_clean = extract_timestamp(data_train_clean)
data_test_clean = extract_timestamp(data_test_clean)

Estandarización de variables cuantitativas

In [11]:
# Standardize numerical variables
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
columns = ['novedad del contrato', 'facturación', 'mora', 'minutos', 'Antigüedad Equipo', 'age']

data_train_clean[columns] = scaler.fit_transform(data_train_clean[columns])
data_test_clean[columns] = scaler.transform(data_test_clean[columns])

In [12]:
# Drop id column
data_train_clean = data_train_clean.drop(columns=['id'])
# save ids in a separate dataframe for later
ids = data_test_clean['id']
data_test_clean = data_test_clean.drop(columns=['id'])

In [13]:
data_train_clean

,Factura online,Antigüedad Equipo,Plan de datos,facturación,mora,novedad del contrato,minutos,resultado,age,hombre,mujer,empresarial
0,1,-1.007978,0,-2.334734,0.754564,1.023311,-0.578726,0,-6.585613,False,True,False
1,0,1.104213,1,0.771149,-0.719395,0.598030,-0.558356,0,-4.542519,False,True,False
2,0,0.906195,1,-0.388154,0.088905,0.514947,-0.155260,1,-5.359756,False,True,False
3,1,-0.875966,1,-0.653097,-0.434113,0.204387,-0.695581,0,-5.768375,False,True,False
4,1,-1.205996,0,1.692216,-0.862036,-1.495751,0.734555,0,-5.768375,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
8238,1,-1.139990,1,-0.746119,0.421734,-0.083921,8.265818,1,-0.660641,False,True,False
8239,1,-0.743954,0,-0.106995,-0.719395,0.282078,-0.559428,0,-0.864950,True,False,False
8240,1,-0.941972,1,0.212065,1.087393,1.054913,0.301441,0,-0.660641,False,True,False
8241,0,0.576165,1,-0.629130,-0.719395,0.631053,-0.739535,0,2.199690,True,False,False


## Entrenamiento del modelo

División de variables predictoras y dependiente y Cross Validation

In [14]:
X = data_train_clean.drop(columns=['resultado'])
Y = data_train_clean['resultado']

from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

Verificar balanceo de datos

In [15]:
# Count number of 0s and 1s percentage in the training set
Y_train.value_counts(normalize=True)

resultado
0    0.835456
1    0.164544
Name: proportion, dtype: float64

### Balanceo de datos

#### SMOTE

In [15]:
# Balance the training set with SMOTE
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train, Y_train = smote.fit_resample(X_train, Y_train)

#### SMOTE + Tomek

In [16]:
# Balance the training set with CNN + Tomek (OSS)

from imblearn.combine import SMOTETomek

smote_tomek = SMOTETomek(random_state=42)

X_train, Y_train = smote_tomek.fit_resample(X_train, Y_train)

### Modelos

#### Regresion logistica

In [36]:
# LOGISTIC REGRESSION
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, Y_train)

LogisticRegression()

#### Arbol de decision CHAID

In [65]:
# MODEL: CHAID DECISION TREE
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(criterion='gini', random_state=42)
model.fit(X_train, Y_train)

DecisionTreeClassifier(random_state=42)

#### KNN

In [67]:
# MODEL: KNN
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, Y_train)

KNeighborsClassifier()

#### SVM

In [39]:
# MODEL: SVM
from sklearn.svm import SVC

model = SVC(kernel='rbf', random_state=42)
model.fit(X_train, Y_train)

SVC(random_state=42)

#### Random Forest Embebido

In [16]:
# MODEL: EMBEDDED RANDOM FOREST
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(min_samples_split=10, max_depth=10, min_samples_leaf=8, n_estimators=400, random_state=42, class_weight=None)
model.fit(X_train, Y_train)

RandomForestClassifier(max_depth=10, min_samples_leaf=8, min_samples_split=10,
                       n_estimators=400, random_state=42)

#### Random Forest con primer intento en parametros (Posible Overfitting)

In [45]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

# Configuración inicial del modelo
initial_model = RandomForestClassifier(random_state=42)

# Definir el espacio de búsqueda inicial para RandomizedSearchCV
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

# Primer paso de búsqueda: una búsqueda aleatoria con RandomizedSearchCV
random_search = RandomizedSearchCV(
    initial_model,
    param_distributions=param_dist,
    n_iter=20,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=-1
)

# Entrenar el primer paso de búsqueda aleatoria
random_search.fit(X_train, Y_train)

# Mejor estimador encontrado en el primer paso
best_model = random_search.best_estimator_

# Ajuste adicional "stepwise" al refinar el espacio de búsqueda en torno a los mejores parámetros
# Extraemos los mejores parámetros y creamos un nuevo rango de búsqueda más centrado
best_params = random_search.best_params_

# Actualizar el espacio de búsqueda centrado en torno a los mejores parámetros
param_dist_refined = {
    'n_estimators': [best_params['n_estimators'] - 50, best_params['n_estimators'], best_params['n_estimators'] + 50],
    'max_depth': [best_params['max_depth'] - 5 if best_params['max_depth'] else None, best_params['max_depth'], 
                  best_params['max_depth'] + 5 if best_params['max_depth'] else None],
    'min_samples_split': [best_params['min_samples_split'] - 1, best_params['min_samples_split'], best_params['min_samples_split'] + 1]
}

# Configurar la búsqueda refinada con RandomizedSearchCV
refined_search = RandomizedSearchCV(
    best_model,
    param_distributions=param_dist_refined,
    n_iter=10,  # Menor número de combinaciones, ajustando alrededor de los mejores parámetros
    scoring='recall',  # Cambiar el criterio de optimización a recall para mejorar el balance de clases
    cv=3,
    random_state=42,
    n_jobs=-1
)

# Entrenar la búsqueda refinada de hiperparámetros
refined_search.fit(X_train, Y_train)

# Mejor modelo encontrado después del ajuste "stepwise"
model = refined_search.best_estimator_

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
9 fits failed out of a total of 30.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
9 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py", line 1466, in wrapper
    estimator._validate_params()
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py", lin

### Random Forest con hiperparametros ajustados para evitar Overfitting

In [21]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

# Definir el modelo base con class_weight='balanced' para manejar desequilibrio
model = RandomForestClassifier(random_state=42, class_weight='balanced')

# Ajuste progresivo de parámetros para reducir el sobreajuste
param_dist = {
    'n_estimators': [100, 150, 200],  # Reducimos el número de estimadores
    'max_depth': [10, 15, 20],  # Aumentamos ligeramente la profundidad
    'min_samples_split': [10, 15, 20],  # Aumentamos para reducir sobreajuste
    'min_samples_leaf': [5, 10, 15],  # Aumentamos el mínimo de muestras en hojas
    'max_features': ['sqrt', 'log2']  # Usamos solo valores válidos para max_features
}

# Configurar RandomizedSearchCV con validación cruzada aumentada
random_search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=20,  # Número de combinaciones a probar
    scoring='roc_auc',  # Optimizar para AUC
    cv=5,  # Validación cruzada con 5 folds para mejor generalización
    random_state=42,
    n_jobs=-1
)

# Entrenar la búsqueda aleatoria de hiperparámetros
random_search.fit(X_train, Y_train)

# Mejor modelo después de la búsqueda
model = random_search.best_estimator_

# Imprimir los mejores parámetros encontrados
print("Best parameters:", random_search.best_params_)
print("Best score:", random_search.best_score_)
print("Best estimator:", random_search.best_estimator_)


Best parameters: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 'log2', 'max_depth': 10}
Best score: 0.8505177701188471
Best estimator: RandomForestClassifier(class_weight='balanced', max_depth=10,
                       max_features='log2', min_samples_leaf=5,
                       min_samples_split=10, n_estimators=200, random_state=42)


#### Embedding con el Random Forest con mejor resultado y un GradientBoosting (Aprender desde el error)

In [22]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

# Definir el modelo de RandomForest con class_weight='balanced' para manejar desequilibrio
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')

# Ajuste progresivo de parámetros para reducir el sobreajuste en RandomForest
param_dist_rf = {
    'n_estimators': [100, 150],  # Reducción en el número de estimadores para evitar sobreajuste
    'max_depth': [8, 10, 15],  # Reducir la profundidad para simplificar el modelo
    'min_samples_split': [15, 20, 30],  # Incremento para reducir la complejidad
    'min_samples_leaf': [10, 15, 20],  # Aumentar para limitar el tamaño del árbol
    'max_features': ['sqrt', 'log2']  # Usar solo valores válidos para max_features
}

# Configurar RandomizedSearchCV para RandomForest
random_search_rf = RandomizedSearchCV(
    rf_model,
    param_distributions=param_dist_rf,
    n_iter=15,  # Menor número de combinaciones a probar
    scoring='roc_auc',  # Optimizar para AUC
    cv=5,  # Validación cruzada con 5 folds para mejor generalización
    random_state=42,
    n_jobs=-1
)

# Entrenar la búsqueda aleatoria de hiperparámetros para RandomForest
random_search_rf.fit(X_train, Y_train)
best_rf_model = random_search_rf.best_estimator_

# Definir el modelo de GradientBoosting con parámetros simplificados
gb_model = GradientBoostingClassifier(random_state=42)

# Ajuste progresivo de parámetros para GradientBoosting
param_dist_gb = {
    'n_estimators': [50, 100],  # Reducir el número de estimadores
    'max_depth': [3, 5],  # Limitar la profundidad
    'learning_rate': [0.01, 0.05],  # Tasa de aprendizaje baja para mayor estabilidad
    'min_samples_split': [15, 20, 30]  # Ajuste para reducir sobreajuste
}

# Configurar RandomizedSearchCV para GradientBoosting
random_search_gb = RandomizedSearchCV(
    gb_model,
    param_distributions=param_dist_gb,
    n_iter=15,  # Menor número de combinaciones a probar
    scoring='roc_auc',  # Optimizar para AUC
    cv=5,  # Validación cruzada
    random_state=42,
    n_jobs=-1
)

# Entrenar la búsqueda aleatoria de hiperparámetros para GradientBoosting
random_search_gb.fit(X_train, Y_train)
best_gb_model = random_search_gb.best_estimator_

# Crear el modelo de conjunto con VotingClassifier usando voto duro
model = VotingClassifier(
    estimators=[('random_forest', best_rf_model), ('gradient_boosting', best_gb_model)],
    voting='hard'  # Cambiar a voto duro para evitar sobreajuste
)

# Entrenar el modelo de conjunto en el conjunto de entrenamiento
model.fit(X_train, Y_train)

# Evaluación en el conjunto de prueba
Y_pred = model.predict(X_test)
Y_pred_proba_rf = best_rf_model.predict_proba(X_test)[:, 1]
Y_pred_proba_gb = best_gb_model.predict_proba(X_test)[:, 1]
# Promedio de probabilidades para VotingClassifier si se desea evaluar AUC
Y_pred_proba_ensemble = (Y_pred_proba_rf + Y_pred_proba_gb) / 2

# Imprimir los mejores parámetros y métricas de rendimiento en el conjunto de prueba
print("Best RandomForest parameters:", random_search_rf.best_params_)
print("Best GradientBoosting parameters:", random_search_gb.best_params_)
print("Voting Classifier accuracy:", accuracy_score(Y_test, Y_pred))
print("Voting Classifier precision:", precision_score(Y_test, Y_pred))
print("Voting Classifier recall:", recall_score(Y_test, Y_pred))
print("Voting Classifier F1:", f1_score(Y_test, Y_pred))
print("Voting Classifier AUC:", roc_auc_score(Y_test, Y_pred_proba_ensemble))



Best RandomForest parameters: {'n_estimators': 150, 'min_samples_split': 30, 'min_samples_leaf': 10, 'max_features': 'sqrt', 'max_depth': 15}
Best GradientBoosting parameters: {'n_estimators': 100, 'min_samples_split': 20, 'max_depth': 5, 'learning_rate': 0.05}
Voting Classifier accuracy: 0.8714372346876895
Voting Classifier precision: 0.7711864406779662
Voting Classifier recall: 0.32971014492753625
Voting Classifier F1: 0.4619289340101523
Voting Classifier AUC: 0.847110949259529


#### Embedding con Arbol de Bosques Aleatorios, Gradient Boosting, KNN, RegLogistica, Voting Soft

In [23]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import RandomOverSampler
import pandas as pd

# Balanceo selectivo: Balancear solo para LogisticRegression y KNeighborsClassifier
oversampler = RandomOverSampler(random_state=42)
X_train_balanced, Y_train_balanced = oversampler.fit_resample(X_train, Y_train)

# RandomForest sin balanceo explícito, con class_weight='balanced'
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
param_dist_rf = {
    'n_estimators': [100, 150, 200, 220],
    'max_depth': [8, 10, 15, 18],
    'min_samples_split': [10, 15, 20],
    'min_samples_leaf': [5, 7, 10, 12, 15],
    'max_features': ['sqrt', 'log2']
}
random_search_rf = RandomizedSearchCV(
    rf_model,
    param_distributions=param_dist_rf,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_rf.fit(X_train, Y_train)
best_rf_model = random_search_rf.best_estimator_

# GradientBoosting sin balanceo explícito
gb_model = GradientBoostingClassifier(random_state=42)
param_dist_gb = {
    'n_estimators': [100, 110, 120, 130, 140, 150],
    'max_depth': [3, 5, 7, 9, 10],
    'learning_rate': [0.05, 0.06, 0.07, 0.08, 0.09, 0.1],
    'min_samples_split': [10, 15, 20, 25, 30]
}
random_search_gb = RandomizedSearchCV(
    gb_model,
    param_distributions=param_dist_gb,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_gb.fit(X_train, Y_train)
best_gb_model = random_search_gb.best_estimator_

# LogisticRegression con balanceo de datos
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
param_dist_logistic = {
    'C': [0.01 ,0.05, 0.07, 0.1, 0.3, 0.5, 0.7, 1, 10, 100]
}
random_search_logistic = RandomizedSearchCV(
    logistic_model,
    param_distributions=param_dist_logistic,
    n_iter=5,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_logistic.fit(X_train_balanced, Y_train_balanced)
best_logistic_model = random_search_logistic.best_estimator_

# KNeighborsClassifier con balanceo de datos
knn_model = KNeighborsClassifier()
param_dist_knn = {
    'n_neighbors': [20, 25, 30, 35, 40, 45],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}
random_search_knn = RandomizedSearchCV(
    knn_model,
    param_distributions=param_dist_knn,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1,
    
)
random_search_knn.fit(X_train_balanced, Y_train_balanced)
best_knn_model = random_search_knn.best_estimator_

# Crear el modelo de conjunto con VotingClassifier usando voto suave
model = VotingClassifier(
    estimators=[
        ('random_forest', best_rf_model),
        ('gradient_boosting', best_gb_model),
        ('logistic', best_logistic_model),
        ('knn', best_knn_model)
    ],
    voting='soft'
)

# Entrenar el modelo de conjunto en el conjunto de entrenamiento
model.fit(X_train_balanced, Y_train_balanced)

# Evaluación en el conjunto de prueba
Y_pred = model.predict(X_test)
Y_pred_proba = model.predict_proba(X_test)[:, 1]

# Imprimir los mejores parámetros y métricas de rendimiento en el conjunto de prueba
print("Best RandomForest parameters:", random_search_rf.best_params_)
print("Best GradientBoosting parameters:", random_search_gb.best_params_)
print("Best Logistic Regression parameters:", random_search_logistic.best_params_)
print("Best KNeighbors parameters:", random_search_knn.best_params_)
print("Voting Classifier accuracy:", accuracy_score(Y_test, Y_pred))
print("Voting Classifier precision:", precision_score(Y_test, Y_pred))
print("Voting Classifier recall:", recall_score(Y_test, Y_pred))
print("Voting Classifier F1:", f1_score(Y_test, Y_pred))
print("Voting Classifier AUC:", roc_auc_score(Y_test, Y_pred_proba))

Best RandomForest parameters: {'n_estimators': 200, 'min_samples_split': 20, 'min_samples_leaf': 10, 'max_features': 'sqrt', 'max_depth': 18}
Best GradientBoosting parameters: {'n_estimators': 130, 'min_samples_split': 15, 'max_depth': 5, 'learning_rate': 0.05}
Best Logistic Regression parameters: {'C': 0.5}
Best KNeighbors parameters: {'weights': 'distance', 'p': 2, 'n_neighbors': 35}
Voting Classifier accuracy: 0.8101879927228623
Voting Classifier precision: 0.4516971279373368
Voting Classifier recall: 0.6268115942028986
Voting Classifier F1: 0.5250379362670713
Voting Classifier AUC: 0.834489164740281


#### Ensamble con GradientBoosting (Con perdida exponencial para penalizar más fuertemente) + RandomForest de buenos resultados

In [30]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import RandomOverSampler
import pandas as pd
import numpy as np

# Balanceo selectivo: Balancear solo para modelos que lo necesiten
oversampler = RandomOverSampler(random_state=42)
X_train_balanced, Y_train_balanced = oversampler.fit_resample(X_train, Y_train)

# Primera etapa: GradientBoostingClassifier sin balanceo explícito
gb_model = GradientBoostingClassifier(random_state=42)
param_dist_gb = {
    'n_estimators': [100, 110, 120, 130, 140, 150],
    'max_depth': [3, 5, 7, 9, 10],
    'learning_rate': [0.05, 0.06, 0.07, 0.08, 0.09, 0.1],
    'min_samples_split': [10, 15, 20, 25, 30],
    'loss' : ['log_loss', 'exponential']
}
random_search_gb = RandomizedSearchCV(
    gb_model,
    param_distributions=param_dist_gb,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_gb.fit(X_train, Y_train)
best_gb_model = random_search_gb.best_estimator_

# Obtener las predicciones (probabilidades) de GradientBoostingClassifier en los datos de entrenamiento y prueba
train_gb_pred = best_gb_model.predict_proba(X_train)[:, 1].reshape(-1, 1)  # Probabilidades para la clase positiva
test_gb_pred = best_gb_model.predict_proba(X_test)[:, 1].reshape(-1, 1)

# Concatenar las predicciones del modelo GradientBoosting con las características originales
X_train_enhanced = np.hstack((X_train, train_gb_pred))
X_test_enhanced = np.hstack((X_test, test_gb_pred))

# Segunda etapa: RandomForestClassifier usando las características mejoradas
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
param_dist_rf = {
    'n_estimators': [100, 150, 200, 220],
    'max_depth': [8, 10, 15, 18],
    'min_samples_split': [10, 15, 20],
    'min_samples_leaf': [5, 7, 10, 12, 15],
    'max_features': ['sqrt', 'log2']
}
random_search_rf = RandomizedSearchCV(
    rf_model,
    param_distributions=param_dist_rf,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_rf.fit(X_train_enhanced, Y_train)
best_rf_model = random_search_rf.best_estimator_

# Evaluación del modelo RandomForest en el conjunto de prueba
Y_pred = best_rf_model.predict(X_test_enhanced)
Y_pred_proba = best_rf_model.predict_proba(X_test_enhanced)[:, 1]

# Imprimir los mejores parámetros y métricas de rendimiento en el conjunto de prueba
print("Best GradientBoosting parameters:", random_search_gb.best_params_)
print("Best RandomForest parameters:", random_search_rf.best_params_)
print("Ensemble Model (GradientBoosting + RandomForest) accuracy:", accuracy_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + RandomForest) precision:", precision_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + RandomForest) recall:", recall_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + RandomForest) F1:", f1_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + RandomForest) AUC:", roc_auc_score(Y_test, Y_pred_proba))


Best GradientBoosting parameters: {'n_estimators': 140, 'min_samples_split': 20, 'max_depth': 3, 'loss': 'exponential', 'learning_rate': 0.06}
Best RandomForest parameters: {'n_estimators': 200, 'min_samples_split': 15, 'min_samples_leaf': 10, 'max_features': 'sqrt', 'max_depth': 8}
Ensemble Model (GradientBoosting + RandomForest) accuracy: 0.8065494238932687
Ensemble Model (GradientBoosting + RandomForest) precision: 0.44941176470588234
Ensemble Model (GradientBoosting + RandomForest) recall: 0.6920289855072463
Ensemble Model (GradientBoosting + RandomForest) F1: 0.5449358059914408
Ensemble Model (GradientBoosting + RandomForest) AUC: 0.8596456505905823


In [46]:
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import RandomOverSampler
import pandas as pd
import numpy as np

# Balanceo selectivo: Balancear solo para modelos que lo necesiten
oversampler = RandomOverSampler(random_state=42)
X_train_balanced, Y_train_balanced = oversampler.fit_resample(X_train, Y_train)

# Primera etapa: XGBClassifier con objective='binary:logistic' para problemas de clasificación binaria
xgb_model = XGBClassifier(objective='binary:logistic', random_state=42, eval_metric='logloss')
param_dist_xgb = {
    'n_estimators': [100, 110, 120, 130, 140, 150],
    'max_depth': [3, 5, 7, 9, 10],
    'learning_rate': [0.05, 0.06, 0.07, 0.08, 0.09, 0.1],
    'min_child_weight': [1, 3, 5],  # Equivalente a min_samples_split en GradientBoosting
    'gamma': [0, 0.1, 0.2, 0.3, 0.4],  # Regularización para XGBoost
    'subsample': [0.7, 0.8, 0.9, 1.0],  # Para controlar el muestreo de filas
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]  # Para controlar el muestreo de columnas
}
random_search_xgb = RandomizedSearchCV(
    xgb_model,
    param_distributions=param_dist_xgb,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_xgb.fit(X_train, Y_train)
best_xgb_model = random_search_xgb.best_estimator_

# Obtener las predicciones (probabilidades) de XGBClassifier en los datos de entrenamiento y prueba
train_xgb_pred = best_xgb_model.predict_proba(X_train)[:, 1].reshape(-1, 1)  # Probabilidades para la clase positiva
test_xgb_pred = best_xgb_model.predict_proba(X_test)[:, 1].reshape(-1, 1)

# Concatenar las predicciones del modelo XGBClassifier con las características originales
X_train_enhanced = np.hstack((X_train, train_xgb_pred))
X_test_enhanced = np.hstack((X_test, test_xgb_pred))

# Segunda etapa: RandomForestClassifier usando las características mejoradas
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
param_dist_rf = {
    'n_estimators': [100, 120, 140, 160, 180, 200],
    'max_depth': [8, 10, 15, 18],
    'min_samples_split': [8, 10, 15],
    'min_samples_leaf': [5, 7, 8 ,9, 10],
    'max_features': ['sqrt', 'log2']
}
random_search_rf = RandomizedSearchCV(
    rf_model,
    param_distributions=param_dist_rf,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_rf.fit(X_train_enhanced, Y_train)
best_rf_model = random_search_rf.best_estimator_

# Evaluación del modelo RandomForest en el conjunto de prueba
Y_pred = best_rf_model.predict(X_test_enhanced)
Y_pred_proba = best_rf_model.predict_proba(X_test_enhanced)[:, 1]

# Imprimir los mejores parámetros y métricas de rendimiento en el conjunto de prueba
print("Best XGBoost parameters:", random_search_xgb.best_params_)
print("Best RandomForest parameters:", random_search_rf.best_params_)
print("Ensemble Model (XGBoost + RandomForest) accuracy:", accuracy_score(Y_test, Y_pred))
print("Ensemble Model (XGBoost + RandomForest) precision:", precision_score(Y_test, Y_pred))
print("Ensemble Model (XGBoost + RandomForest) recall:", recall_score(Y_test, Y_pred))
print("Ensemble Model (XGBoost + RandomForest) F1:", f1_score(Y_test, Y_pred))
print("Ensemble Model (XGBoost + RandomForest) AUC:", roc_auc_score(Y_test, Y_pred_proba))

# Save on submission.csv file the predictions
# submission = pd.DataFrame({'id': ids, 'resultado': Y_pred})
# submission.to_csv('submission.csv', index=False)

Best XGBoost parameters: {'subsample': 1.0, 'n_estimators': 140, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.08, 'gamma': 0.2, 'colsample_bytree': 1.0}
Best RandomForest parameters: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 9, 'max_features': 'sqrt', 'max_depth': 8}
Ensemble Model (XGBoost + RandomForest) accuracy: 0.8180715585203153
Ensemble Model (XGBoost + RandomForest) precision: 0.46954314720812185
Ensemble Model (XGBoost + RandomForest) recall: 0.6702898550724637
Ensemble Model (XGBoost + RandomForest) F1: 0.5522388059701493
Ensemble Model (XGBoost + RandomForest) AUC: 0.8600520388021576


In [47]:
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import RandomOverSampler
import pandas as pd
import numpy as np

# Balanceo selectivo: Balancear solo para modelos que lo necesiten
oversampler = RandomOverSampler(random_state=42)
X_train_balanced, Y_train_balanced = oversampler.fit_resample(X_train, Y_train)

# Primera etapa: XGBClassifier
xgb_model = XGBClassifier(objective='binary:logistic', random_state=42, eval_metric='logloss')
param_dist_xgb = {
    'n_estimators': [100, 110, 120, 130, 140, 150],
    'max_depth': [3, 5, 7, 9, 10],
    'learning_rate': [0.05, 0.06, 0.07, 0.08, 0.09, 0.1],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2, 0.3, 0.4],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}
random_search_xgb = RandomizedSearchCV(
    xgb_model,
    param_distributions=param_dist_xgb,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_xgb.fit(X_train, Y_train)
best_xgb_model = random_search_xgb.best_estimator_

# Obtener las predicciones (probabilidades) de XGBClassifier
train_xgb_pred = best_xgb_model.predict_proba(X_train)[:, 1].reshape(-1, 1)
test_xgb_pred = best_xgb_model.predict_proba(X_test)[:, 1].reshape(-1, 1)

# Concatenar las predicciones del modelo XGBClassifier con las características originales
X_train_enhanced = np.hstack((X_train, train_xgb_pred))
X_test_enhanced = np.hstack((X_test, test_xgb_pred))

# Segunda etapa: RandomForestClassifier con class_weight='balanced'
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
param_dist_rf = {
    'n_estimators': [100, 120, 140, 160, 180, 200],
    'max_depth': [8, 10, 15, 18],
    'min_samples_split': [8, 10, 15],
    'min_samples_leaf': [5, 7, 8, 9, 10],
    'max_features': ['sqrt', 'log2']
}
random_search_rf = RandomizedSearchCV(
    rf_model,
    param_distributions=param_dist_rf,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_rf.fit(X_train_enhanced, Y_train)
best_rf_model = random_search_rf.best_estimator_

# Obtener las predicciones de probabilidad de RandomForestClassifier
train_rf_pred = best_rf_model.predict_proba(X_train_enhanced)[:, 1].reshape(-1, 1)
test_rf_pred = best_rf_model.predict_proba(X_test_enhanced)[:, 1].reshape(-1, 1)

# Concatenar las probabilidades de ambos modelos para Naive Bayes
X_train_final = np.hstack((train_xgb_pred, train_rf_pred))
X_test_final = np.hstack((test_xgb_pred, test_rf_pred))

# Tercera etapa: Naive Bayes
nb_model = GaussianNB()
nb_model.fit(X_train_final, Y_train)

# Evaluación del modelo Naive Bayes en el conjunto de prueba
Y_pred = nb_model.predict(X_test_final)
Y_pred_proba = nb_model.predict_proba(X_test_final)[:, 1]

# Imprimir los mejores parámetros y métricas de rendimiento en el conjunto de prueba
print("Best XGBoost parameters:", random_search_xgb.best_params_)
print("Best RandomForest parameters:", random_search_rf.best_params_)
print("Ensemble Model (XGBoost + RandomForest + Naive Bayes) accuracy:", accuracy_score(Y_test, Y_pred))
print("Ensemble Model (XGBoost + RandomForest + Naive Bayes) precision:", precision_score(Y_test, Y_pred))
print("Ensemble Model (XGBoost + RandomForest + Naive Bayes) recall:", recall_score(Y_test, Y_pred))
print("Ensemble Model (XGBoost + RandomForest + Naive Bayes) F1:", f1_score(Y_test, Y_pred))
print("Ensemble Model (XGBoost + RandomForest + Naive Bayes) AUC:", roc_auc_score(Y_test, Y_pred_proba))

# Guardar las predicciones en un archivo CSV
# submission = pd.DataFrame({'id': ids, 'resultado': Y_pred})
# submission.to_csv('submission.csv', index=False)


Best XGBoost parameters: {'subsample': 1.0, 'n_estimators': 140, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.08, 'gamma': 0.2, 'colsample_bytree': 1.0}
Best RandomForest parameters: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 9, 'max_features': 'sqrt', 'max_depth': 8}
Ensemble Model (XGBoost + RandomForest + Naive Bayes) accuracy: 0.8617343844754397
Ensemble Model (XGBoost + RandomForest + Naive Bayes) precision: 0.5845070422535211
Ensemble Model (XGBoost + RandomForest + Naive Bayes) recall: 0.6014492753623188
Ensemble Model (XGBoost + RandomForest + Naive Bayes) F1: 0.5928571428571429
Ensemble Model (XGBoost + RandomForest + Naive Bayes) AUC: 0.8603475938651213


In [49]:
from sklearn.ensemble import GradientBoostingClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import numpy as np

# Primera etapa: GradientBoostingClassifier
gb_model = GradientBoostingClassifier(random_state=42)
param_dist_gb = {
    'n_estimators': [100, 110, 120, 130, 140, 150],
    'max_depth': [3, 5, 7, 9, 10],
    'learning_rate': [0.05, 0.06, 0.07, 0.08, 0.09, 0.1],
    'min_samples_split': [10, 15, 20, 25, 30],
    'loss' : ['log_loss', 'exponential']
}
random_search_gb = RandomizedSearchCV(
    gb_model,
    param_distributions=param_dist_gb,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_gb.fit(X_train, Y_train)
best_gb_model = random_search_gb.best_estimator_

# Obtener las predicciones (probabilidades) de GradientBoostingClassifier en los datos de entrenamiento y prueba
train_gb_pred = best_gb_model.predict_proba(X_train)[:, 1].reshape(-1, 1)  # Probabilidades para la clase positiva
test_gb_pred = best_gb_model.predict_proba(X_test)[:, 1].reshape(-1, 1)

# Concatenar las predicciones del modelo GradientBoosting con las características originales
X_train_enhanced = np.hstack((X_train, train_gb_pred))
X_test_enhanced = np.hstack((X_test, test_gb_pred))

# Segunda etapa: BalancedRandomForestClassifier usando las características mejoradas
brf_model = BalancedRandomForestClassifier(random_state=42)
param_dist_brf = {
    'n_estimators': [100, 150, 200, 220],
    'max_depth': [8, 10, 15, 18],
    'min_samples_split': [10, 15, 20],
    'min_samples_leaf': [5, 7, 10, 12, 15],
    'max_features': ['sqrt', 'log2']
}
random_search_brf = RandomizedSearchCV(
    brf_model,
    param_distributions=param_dist_brf,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_brf.fit(X_train_enhanced, Y_train)
best_brf_model = random_search_brf.best_estimator_

# Evaluación del modelo ensamble en el conjunto de prueba
Y_pred = best_brf_model.predict(X_test_enhanced)
Y_pred_proba = best_brf_model.predict_proba(X_test_enhanced)[:, 1]

# Imprimir los mejores parámetros y métricas de rendimiento en el conjunto de prueba
print("Best GradientBoosting parameters:", random_search_gb.best_params_)
print("Best BalancedRandomForest parameters:", random_search_brf.best_params_)
print("Ensemble Model (GradientBoosting + BalancedRandomForest) accuracy:", accuracy_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + BalancedRandomForest) precision:", precision_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + BalancedRandomForest) recall:", recall_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + BalancedRandomForest) F1:", f1_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + BalancedRandomForest) AUC:", roc_auc_score(Y_test, Y_pred_proba))

# Guardar las predicciones en un archivo CSV
# submission = pd.DataFrame({'id': ids, 'resultado': Y_pred})
# submission.to_csv('submission.csv', index=False)


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\imblearn\ensemble\_forest.py:577: FutureWarning: The default of `sampling_strategy` will change from `'auto'` to `'all'` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `'all'` to silence this warning and adopt the future behaviour.
  warn(
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\imblearn\ensemble\_forest.py:589: FutureWarning: The default of `replacement` will change from `False` to `True` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `True` to silence this warning and adopt the future behaviour.
  warn(
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\imblearn\ensemble\_forest.py:601: FutureWarning: The default of `bootstrap` will change from `True` to `False` in version 0.13. This change will follow the implementation proposed in the original paper. Set to

Best GradientBoosting parameters: {'n_estimators': 140, 'min_samples_split': 20, 'max_depth': 3, 'loss': 'exponential', 'learning_rate': 0.06}
Best BalancedRandomForest parameters: {'n_estimators': 100, 'min_samples_split': 15, 'min_samples_leaf': 10, 'max_features': 'sqrt', 'max_depth': 15}
Ensemble Model (GradientBoosting + BalancedRandomForest) accuracy: 0.7907822922983626
Ensemble Model (GradientBoosting + BalancedRandomForest) precision: 0.4279749478079332
Ensemble Model (GradientBoosting + BalancedRandomForest) recall: 0.7427536231884058
Ensemble Model (GradientBoosting + BalancedRandomForest) F1: 0.543046357615894
Ensemble Model (GradientBoosting + BalancedRandomForest) AUC: 0.8606167600831776


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import numpy as np

# Primera etapa: GradientBoostingClassifier
gb_model = GradientBoostingClassifier(random_state=42)
param_dist_gb = {
    'n_estimators': [100, 110, 120, 130, 140, 150],
    'max_depth': [3, 5, 7, 9, 10],
    'learning_rate': [0.05, 0.06, 0.07, 0.08, 0.09, 0.1],
    'min_samples_split': [10, 15, 20, 25, 30],
    'loss': ['log_loss', 'exponential']
}
random_search_gb = RandomizedSearchCV(
    gb_model,
    param_distributions=param_dist_gb,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_gb.fit(X_train, Y_train)
best_gb_model = random_search_gb.best_estimator_

# Obtener las predicciones (probabilidades) de GradientBoostingClassifier en los datos de entrenamiento y prueba
train_gb_pred = best_gb_model.predict_proba(X_train)[:, 1].reshape(-1, 1)  # Probabilidades para la clase positiva
test_gb_pred = best_gb_model.predict_proba(X_test)[:, 1].reshape(-1, 1)

# Concatenar las predicciones del modelo GradientBoosting con las características originales
X_train_enhanced = np.hstack((X_train, train_gb_pred))
X_test_enhanced = np.hstack((X_test, test_gb_pred))

# Segunda etapa: BalancedRandomForestClassifier
brf_model = BalancedRandomForestClassifier(random_state=42)
param_dist_brf = {
    'n_estimators': [100, 150, 200, 220],
    'max_depth': [8, 10, 15, 18],
    'min_samples_split': [10, 15, 20],
    'min_samples_leaf': [5, 7, 10, 12, 15],
    'max_features': ['sqrt', 'log2']
}
random_search_brf = RandomizedSearchCV(
    brf_model,
    param_distributions=param_dist_brf,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_brf.fit(X_train_enhanced, Y_train)
best_brf_model = random_search_brf.best_estimator_

# Obtener las predicciones de probabilidad de BalancedRandomForestClassifier
train_brf_pred = best_brf_model.predict_proba(X_train_enhanced)[:, 1].reshape(-1, 1)
test_brf_pred = best_brf_model.predict_proba(X_test_enhanced)[:, 1].reshape(-1, 1)

# Concatenar las probabilidades de GradientBoosting y BalancedRandomForest para usar en LogisticRegression
X_train_final = np.hstack((X_train_enhanced, train_brf_pred))
X_test_final = np.hstack((X_test_enhanced, test_brf_pred))

# Tercera etapa: LogisticRegression
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
param_dist_lr = {
    'C': [0.1, 0.3, 0.5, 0.8, 1],  # Regularización
    'solver': ['liblinear', 'lbfgs']  # Algoritmos de optimización para ajustarse mejor a diferentes configuraciones
}
random_search_lr = RandomizedSearchCV(
    lr_model,
    param_distributions=param_dist_lr,
    n_iter=10,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_lr.fit(X_train_final, Y_train)
best_lr_model = random_search_lr.best_estimator_

# Evaluación del modelo de regresión logística en el conjunto de prueba
Y_pred = best_lr_model.predict(X_test_final)
Y_pred_proba = best_lr_model.predict_proba(X_test_final)[:, 1]

# Imprimir los mejores parámetros y métricas de rendimiento en el conjunto de prueba
print("Best GradientBoosting parameters:", random_search_gb.best_params_)
print("Best BalancedRandomForest parameters:", random_search_brf.best_params_)
print("Best Logistic Regression parameters:", random_search_lr.best_params_)
print("Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression) accuracy:", accuracy_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression) precision:", precision_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression) recall:", recall_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression) F1:", f1_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression) AUC:", roc_auc_score(Y_test, Y_pred_proba))

# Guardar las predicciones en un archivo CSV
# submission = pd.DataFrame({'id': ids, 'resultado': Y_pred})
# submission.to_csv('submission.csv', index=False)


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\imblearn\ensemble\_forest.py:577: FutureWarning: The default of `sampling_strategy` will change from `'auto'` to `'all'` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `'all'` to silence this warning and adopt the future behaviour.
  warn(
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\imblearn\ensemble\_forest.py:589: FutureWarning: The default of `replacement` will change from `False` to `True` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `True` to silence this warning and adopt the future behaviour.
  warn(
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\imblearn\ensemble\_forest.py:601: FutureWarning: The default of `bootstrap` will change from `True` to `False` in version 0.13. This change will follow the implementation proposed in the original paper. Set to

Best GradientBoosting parameters: {'n_estimators': 140, 'min_samples_split': 20, 'max_depth': 3, 'loss': 'exponential', 'learning_rate': 0.06}
Best BalancedRandomForest parameters: {'n_estimators': 100, 'min_samples_split': 15, 'min_samples_leaf': 10, 'max_features': 'sqrt', 'max_depth': 15}
Best Logistic Regression parameters: {'solver': 'lbfgs', 'C': 0.1}
Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression) accuracy: 0.8047301394784718
Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression) precision: 0.4474885844748858
Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression) recall: 0.7101449275362319
Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression) F1: 0.5490196078431373
Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression) AUC: 0.8596667616665084


In [54]:
from sklearn.ensemble import GradientBoostingClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import numpy as np

# Primera etapa: GradientBoostingClassifier
gb_model = GradientBoostingClassifier(random_state=42)
param_dist_gb = {
    'n_estimators': [100, 110, 120, 130, 140, 150],
    'max_depth': [3, 5, 7, 9, 10],
    'learning_rate': [0.05, 0.06, 0.07, 0.08, 0.09, 0.1],
    'min_samples_split': [10, 15, 20, 25, 30],
    'loss': ['log_loss', 'exponential']
}
random_search_gb = RandomizedSearchCV(
    gb_model,
    param_distributions=param_dist_gb,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_gb.fit(X_train, Y_train)
best_gb_model = random_search_gb.best_estimator_

# Obtener las predicciones (probabilidades) de GradientBoostingClassifier
train_gb_pred = best_gb_model.predict_proba(X_train)[:, 1].reshape(-1, 1)
test_gb_pred = best_gb_model.predict_proba(X_test)[:, 1].reshape(-1, 1)

# Concatenar las predicciones del modelo GradientBoosting con las características originales
X_train_enhanced = np.hstack((X_train, train_gb_pred))
X_test_enhanced = np.hstack((X_test, test_gb_pred))

# Segunda etapa: BalancedRandomForestClassifier
brf_model = BalancedRandomForestClassifier(random_state=42)
param_dist_brf = {
    'n_estimators': [100, 150, 200, 220],
    'max_depth': [8, 10, 15, 18],
    'min_samples_split': [10, 15, 20],
    'min_samples_leaf': [5, 7, 10, 12, 15],
    'max_features': ['sqrt', 'log2']
}
random_search_brf = RandomizedSearchCV(
    brf_model,
    param_distributions=param_dist_brf,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_brf.fit(X_train_enhanced, Y_train)
best_brf_model = random_search_brf.best_estimator_

# Obtener las predicciones de probabilidad de BalancedRandomForestClassifier
train_brf_pred = best_brf_model.predict_proba(X_train_enhanced)[:, 1].reshape(-1, 1)
test_brf_pred = best_brf_model.predict_proba(X_test_enhanced)[:, 1].reshape(-1, 1)

# Concatenar las probabilidades de GradientBoosting y BalancedRandomForest para usar en LogisticRegression
X_train_final = np.hstack((X_train_enhanced, train_brf_pred))
X_test_final = np.hstack((X_test_enhanced, test_brf_pred))

# Tercera etapa: LogisticRegression
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
param_dist_lr = {
    'C': [0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs']
}
random_search_lr = RandomizedSearchCV(
    lr_model,
    param_distributions=param_dist_lr,
    n_iter=10,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_lr.fit(X_train_final, Y_train)
best_lr_model = random_search_lr.best_estimator_

# Obtener las predicciones de probabilidad de LogisticRegression
train_lr_pred = best_lr_model.predict_proba(X_train_final)[:, 1].reshape(-1, 1)
test_lr_pred = best_lr_model.predict_proba(X_test_final)[:, 1].reshape(-1, 1)

# Concatenar las probabilidades de todas las etapas anteriores para usar en Naive Bayes
X_train_final_nb = np.hstack((X_train_final, train_lr_pred))
X_test_final_nb = np.hstack((X_test_final, test_lr_pred))

# Cuarta etapa: Naive Bayes
nb_model = GaussianNB()
nb_model.fit(X_train_final_nb, Y_train)

# Evaluación del modelo Naive Bayes en el conjunto de prueba
Y_pred = nb_model.predict(X_test_final_nb)
Y_pred_proba = nb_model.predict_proba(X_test_final_nb)[:, 1]

# Imprimir los mejores parámetros y métricas de rendimiento en el conjunto de prueba
print("Best GradientBoosting parameters:", random_search_gb.best_params_)
print("Best BalancedRandomForest parameters:", random_search_brf.best_params_)
print("Best Logistic Regression parameters:", random_search_lr.best_params_)
print("Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression + Naive Bayes) accuracy:", accuracy_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression + Naive Bayes) precision:", precision_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression + Naive Bayes) recall:", recall_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression + Naive Bayes) F1:", f1_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression + Naive Bayes) AUC:", roc_auc_score(Y_test, Y_pred_proba))

# Guardar las predicciones en un archivo CSV
# submission = pd.DataFrame({'id': ids, 'resultado': Y_pred})
# submission.to_csv('submission.csv', index=False)


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\imblearn\ensemble\_forest.py:577: FutureWarning: The default of `sampling_strategy` will change from `'auto'` to `'all'` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `'all'` to silence this warning and adopt the future behaviour.
  warn(
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\imblearn\ensemble\_forest.py:589: FutureWarning: The default of `replacement` will change from `False` to `True` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `True` to silence this warning and adopt the future behaviour.
  warn(
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\imblearn\ensemble\_forest.py:601: FutureWarning: The default of `bootstrap` will change from `True` to `False` in version 0.13. This change will follow the implementation proposed in the original paper. Set to

Best GradientBoosting parameters: {'n_estimators': 140, 'min_samples_split': 20, 'max_depth': 3, 'loss': 'exponential', 'learning_rate': 0.06}
Best BalancedRandomForest parameters: {'n_estimators': 100, 'min_samples_split': 15, 'min_samples_leaf': 10, 'max_features': 'sqrt', 'max_depth': 15}
Best Logistic Regression parameters: {'solver': 'liblinear', 'C': 1}
Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression + Naive Bayes) accuracy: 0.8259551243177683
Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression + Naive Bayes) precision: 0.48556430446194226
Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression + Naive Bayes) recall: 0.6702898550724637
Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression + Naive Bayes) F1: 0.563165905631659
Ensemble Model (GradientBoosting + BalancedRandomForest + LogisticRegression + Naive Bayes) AUC: 0.8588038464380336


In [55]:
from sklearn.ensemble import GradientBoostingClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import numpy as np

# Primera etapa: GradientBoostingClassifier
gb_model = GradientBoostingClassifier(random_state=42)
param_dist_gb = {
    'n_estimators': [100, 110, 120, 130, 140, 150],
    'max_depth': [3, 5, 7, 9, 10],
    'learning_rate': [0.05, 0.06, 0.07, 0.08, 0.09, 0.1],
    'min_samples_split': [10, 15, 20, 25, 30],
    'loss': ['log_loss', 'exponential']
}
random_search_gb = RandomizedSearchCV(
    gb_model,
    param_distributions=param_dist_gb,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_gb.fit(X_train, Y_train)
best_gb_model = random_search_gb.best_estimator_

# Obtener las predicciones (probabilidades) de GradientBoostingClassifier
train_gb_pred = best_gb_model.predict_proba(X_train)[:, 1].reshape(-1, 1)
test_gb_pred = best_gb_model.predict_proba(X_test)[:, 1].reshape(-1, 1)

# Concatenar las predicciones del modelo GradientBoosting con las características originales
X_train_enhanced = np.hstack((X_train, train_gb_pred))
X_test_enhanced = np.hstack((X_test, test_gb_pred))

# Segunda etapa: BalancedRandomForestClassifier
brf_model = BalancedRandomForestClassifier(random_state=42)
param_dist_brf = {
    'n_estimators': [100, 150, 200, 220],
    'max_depth': [8, 10, 15, 18],
    'min_samples_split': [10, 15, 20],
    'min_samples_leaf': [5, 7, 10, 12, 15],
    'max_features': ['sqrt', 'log2']
}
random_search_brf = RandomizedSearchCV(
    brf_model,
    param_distributions=param_dist_brf,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_brf.fit(X_train_enhanced, Y_train)
best_brf_model = random_search_brf.best_estimator_

# Obtener las predicciones de probabilidad de BalancedRandomForestClassifier
train_brf_pred = best_brf_model.predict_proba(X_train_enhanced)[:, 1].reshape(-1, 1)
test_brf_pred = best_brf_model.predict_proba(X_test_enhanced)[:, 1].reshape(-1, 1)

# Concatenar las probabilidades de GradientBoosting y BalancedRandomForest para usar en Naive Bayes
X_train_final = np.hstack((X_train_enhanced, train_brf_pred))
X_test_final = np.hstack((X_test_enhanced, test_brf_pred))

# Tercera etapa: Naive Bayes
nb_model = GaussianNB()
nb_model.fit(X_train_final, Y_train)

# Evaluación del modelo Naive Bayes en el conjunto de prueba
Y_pred = nb_model.predict(X_test_final)
Y_pred_proba = nb_model.predict_proba(X_test_final)[:, 1]

# Imprimir los mejores parámetros y métricas de rendimiento en el conjunto de prueba
print("Best GradientBoosting parameters:", random_search_gb.best_params_)
print("Best BalancedRandomForest parameters:", random_search_brf.best_params_)
print("Ensemble Model (GradientBoosting + BalancedRandomForest + Naive Bayes) accuracy:", accuracy_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + BalancedRandomForest + Naive Bayes) precision:", precision_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + BalancedRandomForest + Naive Bayes) recall:", recall_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + BalancedRandomForest + Naive Bayes) F1:", f1_score(Y_test, Y_pred))
print("Ensemble Model (GradientBoosting + BalancedRandomForest + Naive Bayes) AUC:", roc_auc_score(Y_test, Y_pred_proba))

# Guardar las predicciones en un archivo CSV
# submission = pd.DataFrame({'id': ids, 'resultado': Y_pred})
# submission.to_csv('submission.csv', index=False)


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\imblearn\ensemble\_forest.py:577: FutureWarning: The default of `sampling_strategy` will change from `'auto'` to `'all'` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `'all'` to silence this warning and adopt the future behaviour.
  warn(
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\imblearn\ensemble\_forest.py:589: FutureWarning: The default of `replacement` will change from `False` to `True` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `True` to silence this warning and adopt the future behaviour.
  warn(
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\imblearn\ensemble\_forest.py:601: FutureWarning: The default of `bootstrap` will change from `True` to `False` in version 0.13. This change will follow the implementation proposed in the original paper. Set to

Best GradientBoosting parameters: {'n_estimators': 140, 'min_samples_split': 20, 'max_depth': 3, 'loss': 'exponential', 'learning_rate': 0.06}
Best BalancedRandomForest parameters: {'n_estimators': 100, 'min_samples_split': 15, 'min_samples_leaf': 10, 'max_features': 'sqrt', 'max_depth': 15}
Ensemble Model (GradientBoosting + BalancedRandomForest + Naive Bayes) accuracy: 0.8441479684657368
Ensemble Model (GradientBoosting + BalancedRandomForest + Naive Bayes) precision: 0.5295950155763239
Ensemble Model (GradientBoosting + BalancedRandomForest + Naive Bayes) recall: 0.6159420289855072
Ensemble Model (GradientBoosting + BalancedRandomForest + Naive Bayes) F1: 0.5695142378559463
Ensemble Model (GradientBoosting + BalancedRandomForest + Naive Bayes) AUC: 0.8559221845741368


#### Ensamble con GradientBoosting y luego RF

In [ ]:
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import RandomOverSampler
import pandas as pd
import numpy as np

# Balanceo selectivo: Balancear solo para modelos que lo necesiten
oversampler = RandomOverSampler(random_state=42)
X_train_balanced, Y_train_balanced = oversampler.fit_resample(X_train, Y_train)

# Primera etapa: XGBClassifier con objective='binary:logistic' para problemas de clasificación binaria
xgb_model = XGBClassifier(objective='binary:logistic', random_state=42, eval_metric='logloss')
param_dist_xgb = {
    'n_estimators': [100, 110, 120, 130, 140, 150],
    'max_depth': [3, 5, 7, 9, 10],
    'learning_rate': [0.05, 0.06, 0.07, 0.08, 0.09, 0.1],
    'min_child_weight': [1, 3, 5],  # Equivalente a min_samples_split en GradientBoosting
    'gamma': [0, 0.1, 0.2, 0.3, 0.4],  # Regularización para XGBoost
    'subsample': [0.7, 0.8, 0.9, 1.0],  # Para controlar el muestreo de filas
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]  # Para controlar el muestreo de columnas
}
random_search_xgb = RandomizedSearchCV(
    xgb_model,
    param_distributions=param_dist_xgb,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_xgb.fit(X_train, Y_train)
best_xgb_model = random_search_xgb.best_estimator_

# Obtener las predicciones (probabilidades) de XGBClassifier en los datos de entrenamiento y prueba
train_xgb_pred = best_xgb_model.predict_proba(X_train)[:, 1].reshape(-1, 1)  # Probabilidades para la clase positiva
test_xgb_pred = best_xgb_model.predict_proba(X_test)[:, 1].reshape(-1, 1)

# Concatenar las predicciones del modelo XGBClassifier con las características originales
X_train_enhanced = np.hstack((X_train, train_xgb_pred))
X_test_enhanced = np.hstack((X_test, test_xgb_pred))

# Segunda etapa: RandomForestClassifier usando las características mejoradas
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
param_dist_rf = {
    'n_estimators': [100, 120, 140, 160, 180, 200],
    'max_depth': [8, 10, 15, 18],
    'min_samples_split': [8, 10, 15],
    'min_samples_leaf': [5, 7, 8 ,9, 10],
    'max_features': ['sqrt', 'log2']
}
random_search_rf = RandomizedSearchCV(
    rf_model,
    param_distributions=param_dist_rf,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_rf.fit(X_train_enhanced, Y_train)
best_rf_model = random_search_rf.best_estimator_

# Evaluación del modelo RandomForest en el conjunto de prueba
Y_pred = best_rf_model.predict(X_test_enhanced)
Y_pred_proba = best_rf_model.predict_proba(X_test_enhanced)[:, 1]

# Imprimir los mejores parámetros y métricas de rendimiento en el conjunto de prueba
print("Best XGBoost parameters:", random_search_xgb.best_params_)
print("Best RandomForest parameters:", random_search_rf.best_params_)
print("Ensemble Model (XGBoost + RandomForest) accuracy:", accuracy_score(Y_test, Y_pred))
print("Ensemble Model (XGBoost + RandomForest) precision:", precision_score(Y_test, Y_pred))
print("Ensemble Model (XGBoost + RandomForest) recall:", recall_score(Y_test, Y_pred))
print("Ensemble Model (XGBoost + RandomForest) F1:", f1_score(Y_test, Y_pred))
print("Ensemble Model (XGBoost + RandomForest) AUC:", roc_auc_score(Y_test, Y_pred_proba))

# Save on submission.csv file the predictions
# submission = pd.DataFrame({'id': ids, 'resultado': Y_pred})
# submission.to_csv('submission.csv', index=False)


Best XGBoost parameters: {'subsample': 0.9, 'n_estimators': 150, 'min_child_weight': 5, 'max_depth': 10, 'learning_rate': 0.08, 'gamma': 0.2, 'colsample_bytree': 1.0}
Best RandomForest parameters: {'n_estimators': 180, 'min_samples_split': 8, 'min_samples_leaf': 10, 'max_features': 'log2', 'max_depth': 18}
Ensemble Model (XGBoost + RandomForest) accuracy: 0.8435415403274712
Ensemble Model (XGBoost + RandomForest) precision: 0.5294117647058824
Ensemble Model (XGBoost + RandomForest) recall: 0.5869565217391305
Ensemble Model (XGBoost + RandomForest) F1: 0.5567010309278351
Ensemble Model (XGBoost + RandomForest) AUC: 0.8343255539018546


In [ ]:
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import RandomOverSampler
import pandas as pd
import numpy as np

# Balanceo selectivo: Balancear solo para modelos que lo necesiten
oversampler = RandomOverSampler(random_state=42)
X_train_balanced, Y_train_balanced = oversampler.fit_resample(X_train, Y_train)

# Primera etapa: XGBClassifier con objective='binary:logistic' para problemas de clasificación binaria
xgb_model = XGBClassifier(objective='binary:logistic', random_state=42, use_label_encoder=False, eval_metric='logloss')
param_dist_xgb = {
    'n_estimators': [100, 110, 120, 130, 140, 150],
    'max_depth': [3, 5, 7, 9, 10],
    'learning_rate': [0.05, 0.06, 0.07, 0.08, 0.09, 0.1],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2, 0.3, 0.4],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}
random_search_xgb = RandomizedSearchCV(
    xgb_model,
    param_distributions=param_dist_xgb,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_xgb.fit(X_train, Y_train)
best_xgb_model = random_search_xgb.best_estimator_

# Obtener las predicciones (probabilidades) de XGBClassifier en los datos de entrenamiento y prueba
train_xgb_pred = best_xgb_model.predict_proba(X_train)[:, 1].reshape(-1, 1)
test_xgb_pred = best_xgb_model.predict_proba(X_test)[:, 1].reshape(-1, 1)

# Concatenar las predicciones del modelo XGBClassifier con las características originales
X_train_enhanced = np.hstack((X_train, train_xgb_pred))
X_test_enhanced = np.hstack((X_test, test_xgb_pred))

# Segunda etapa: RandomForestClassifier usando las características mejoradas
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
param_dist_rf = {
    'n_estimators': [100, 120, 140, 160, 180, 200],
    'max_depth': [8, 10, 15, 18],
    'min_samples_split': [8, 10, 15],
    'min_samples_leaf': [5, 7, 8, 9, 10],
    'max_features': ['sqrt', 'log2']
}
random_search_rf = RandomizedSearchCV(
    rf_model,
    param_distributions=param_dist_rf,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_rf.fit(X_train_enhanced, Y_train)
best_rf_model = random_search_rf.best_estimator_

# Obtener las predicciones (probabilidades) de RandomForestClassifier
train_rf_pred = best_rf_model.predict_proba(X_train_enhanced)[:, 1].reshape(-1, 1)
test_rf_pred = best_rf_model.predict_proba(X_test_enhanced)[:, 1].reshape(-1, 1)

# Concatenar las predicciones del modelo RandomForest con las características mejoradas para la etapa final
X_train_final = np.hstack((X_train_enhanced, train_rf_pred))
X_test_final = np.hstack((X_test_enhanced, test_rf_pred))

# Tercera etapa: Regresión Logística como modelo final
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
param_dist_logistic = {
    'C': [0.01, 0.05, 0.1, 0.3, 0.5, 0.7, 1, 10, 100]
}
random_search_logistic = RandomizedSearchCV(
    logistic_model,
    param_distributions=param_dist_logistic,
    n_iter=10,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_logistic.fit(X_train_final, Y_train)
best_logistic_model = random_search_logistic.best_estimator_

# Evaluación del modelo final en el conjunto de prueba
Y_pred = best_logistic_model.predict(X_test_final)
Y_pred_proba = best_logistic_model.predict_proba(X_test_final)[:, 1]

# Imprimir los mejores parámetros y métricas de rendimiento en el conjunto de prueba
print("Best XGBoost parameters:", random_search_xgb.best_params_)
print("Best RandomForest parameters:", random_search_rf.best_params_)
print("Best Logistic Regression parameters:", random_search_logistic.best_params_)
print("Ensemble Model (XGBoost + RandomForest + Logistic Regression) accuracy:", accuracy_score(Y_test, Y_pred))
print("Ensemble Model (XGBoost + RandomForest + Logistic Regression) precision:", precision_score(Y_test, Y_pred))
print("Ensemble Model (XGBoost + RandomForest + Logistic Regression) recall:", recall_score(Y_test, Y_pred))
print("Ensemble Model (XGBoost + RandomForest + Logistic Regression) F1:", f1_score(Y_test, Y_pred))
print("Ensemble Model (XGBoost + RandomForest + Logistic Regression) AUC:", roc_auc_score(Y_test, Y_pred_proba))


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:158: UserWarning: [16:12:56] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0ed59c031377d09b8-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 9 is smaller than n_iter=10. Running 9 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best XGBoost parameters: {'subsample': 1.0, 'n_estimators': 140, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.08, 'gamma': 0.2, 'colsample_bytree': 1.0}
Best RandomForest parameters: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 9, 'max_features': 'sqrt', 'max_depth': 8}
Best Logistic Regression parameters: {'C': 100}
Ensemble Model (XGBoost + RandomForest + Logistic Regression) accuracy: 0.8702243784111583
Ensemble Model (XGBoost + RandomForest + Logistic Regression) precision: 0.628099173553719
Ensemble Model (XGBoost + RandomForest + Logistic Regression) recall: 0.5507246376811594
Ensemble Model (XGBoost + RandomForest + Logistic Regression) F1: 0.5868725868725869
Ensemble Model (XGBoost + RandomForest + Logistic Regression) AUC: 0.8498474724764347


In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import RandomOverSampler
import pandas as pd

# Balanceo selectivo: Balancear solo para LogisticRegression y KNeighborsClassifier
oversampler = RandomOverSampler(random_state=42)
X_train_balanced, Y_train_balanced = oversampler.fit_resample(X_train, Y_train)

# RandomForest sin balanceo explícito, con class_weight='balanced'
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
param_dist_rf = {
    'n_estimators': [100, 150, 200, 220],
    'max_depth': [8, 10, 15, 18],
    'min_samples_split': [10, 15, 20],
    'min_samples_leaf': [5, 7, 10, 12, 15]
}
random_search_rf = RandomizedSearchCV(
    rf_model,
    param_distributions=param_dist_rf,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_rf.fit(X_train, Y_train)
best_rf_model = random_search_rf.best_estimator_


# LogisticRegression con balanceo de datos
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
param_dist_logistic = {
    'C': [0.01 ,0.05, 0.07, 0.1, 0.3, 0.5, 0.7, 1, 10, 100]
}
random_search_logistic = RandomizedSearchCV(
    logistic_model,
    param_distributions=param_dist_logistic,
    n_iter=5,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_logistic.fit(X_train_balanced, Y_train_balanced)
best_logistic_model = random_search_logistic.best_estimator_

# KNeighborsClassifier con balanceo de datos
knn_model = KNeighborsClassifier()
param_dist_knn = {
    'n_neighbors': [20, 25, 30, 35, 40, 45],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}
random_search_knn = RandomizedSearchCV(
    knn_model,
    param_distributions=param_dist_knn,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_knn.fit(X_train_balanced, Y_train_balanced)
best_knn_model = random_search_knn.best_estimator_

# Crear el modelo de conjunto con VotingClassifier usando voto suave
model = VotingClassifier(
    estimators=[
        ('random_forest', best_rf_model),
        ('logistic', best_logistic_model),
        ('knn', best_knn_model)
    ],
    voting='soft'
)

# Entrenar el modelo de conjunto en el conjunto de entrenamiento
model.fit(X_train_balanced, Y_train_balanced)

# Evaluación en el conjunto de prueba
Y_pred = model.predict(X_test)
Y_pred_proba = model.predict_proba(X_test)[:, 1]

# Imprimir los mejores parámetros y métricas de rendimiento en el conjunto de prueba
print("Best RandomForest parameters:", random_search_rf.best_params_)
print("Best Logistic Regression parameters:", random_search_logistic.best_params_)
print("Best KNeighbors parameters:", random_search_knn.best_params_)
print("Voting Classifier accuracy:", accuracy_score(Y_test, Y_pred))
print("Voting Classifier precision:", precision_score(Y_test, Y_pred))
print("Voting Classifier recall:", recall_score(Y_test, Y_pred))
print("Voting Classifier F1:", f1_score(Y_test, Y_pred))
print("Voting Classifier AUC:", roc_auc_score(Y_test, Y_pred_proba))



Best RandomForest parameters: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 10, 'max_depth': 10}
Best Logistic Regression parameters: {'C': 0.5}
Best KNeighbors parameters: {'weights': 'distance', 'p': 2, 'n_neighbors': 35}
Voting Classifier accuracy: 0.7871437234687689
Voting Classifier precision: 0.41092636579572445
Voting Classifier recall: 0.6268115942028986
Voting Classifier F1: 0.4964131994261119
Voting Classifier AUC: 0.8094329565006281


#### Embedding de arriba con votación hard

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import RandomOverSampler
import pandas as pd

# Balanceo selectivo: Balancear solo para LogisticRegression y KNeighborsClassifier
oversampler = RandomOverSampler(random_state=42)
X_train_balanced, Y_train_balanced = oversampler.fit_resample(X_train, Y_train)

# RandomForest sin balanceo explícito, con class_weight='balanced'
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
param_dist_rf = {
    'n_estimators': [100, 150, 200, 220],
    'max_depth': [8, 10, 15, 18],
    'min_samples_split': [10, 15, 20],
    'min_samples_leaf': [5, 7, 10, 12, 15]
}
random_search_rf = RandomizedSearchCV(
    rf_model,
    param_distributions=param_dist_rf,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_rf.fit(X_train, Y_train)
best_rf_model = random_search_rf.best_estimator_

# GradientBoosting sin balanceo explícito
gb_model = GradientBoostingClassifier(random_state=42)
param_dist_gb = {
    'n_estimators': [100, 110, 120, 130, 140, 150],
    'max_depth': [3, 5, 7, 9, 10],
    'learning_rate': [0.05, 0.06, 0.07, 0.08, 0.09, 0.1],
    'min_samples_split': [10, 15, 20, 25, 30]
}
random_search_gb = RandomizedSearchCV(
    gb_model,
    param_distributions=param_dist_gb,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_gb.fit(X_train, Y_train)
best_gb_model = random_search_gb.best_estimator_

# LogisticRegression con balanceo de datos
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
param_dist_logistic = {
    'C': [0.01, 0.05, 0.07, 0.1, 0.3, 0.5, 0.7, 1, 10, 100]
}
random_search_logistic = RandomizedSearchCV(
    logistic_model,
    param_distributions=param_dist_logistic,
    n_iter=5,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_logistic.fit(X_train_balanced, Y_train_balanced)
best_logistic_model = random_search_logistic.best_estimator_

# KNeighborsClassifier con balanceo de datos
knn_model = KNeighborsClassifier()
param_dist_knn = {
    'n_neighbors': [20, 25, 30, 35, 40, 45],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}
random_search_knn = RandomizedSearchCV(
    knn_model,
    param_distributions=param_dist_knn,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)
random_search_knn.fit(X_train_balanced, Y_train_balanced)
best_knn_model = random_search_knn.best_estimator_

# Crear el modelo de conjunto con VotingClassifier usando voto duro
model = VotingClassifier(
    estimators=[
        ('random_forest', best_rf_model),
        ('gradient_boosting', best_gb_model),
        ('logistic', best_logistic_model),
        ('knn', best_knn_model)
    ],
    voting='hard'  # Cambiamos a voto duro
)

# Entrenar el modelo de conjunto en el conjunto de entrenamiento
model.fit(X_train_balanced, Y_train_balanced)

# Evaluación en el conjunto de prueba
Y_pred = model.predict(X_test)

# Imprimir los mejores parámetros y métricas de rendimiento en el conjunto de prueba
print("Best RandomForest parameters:", random_search_rf.best_params_)
print("Best GradientBoosting parameters:", random_search_gb.best_params_)
print("Best Logistic Regression parameters:", random_search_logistic.best_params_)
print("Best KNeighbors parameters:", random_search_knn.best_params_)
print("Voting Classifier accuracy:", accuracy_score(Y_test, Y_pred))
print("Voting Classifier precision:", precision_score(Y_test, Y_pred))
print("Voting Classifier recall:", recall_score(Y_test, Y_pred))
print("Voting Classifier F1:", f1_score(Y_test, Y_pred))

Best RandomForest parameters: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 10, 'max_depth': 10}
Best GradientBoosting parameters: {'n_estimators': 130, 'min_samples_split': 15, 'max_depth': 5, 'learning_rate': 0.05}
Best Logistic Regression parameters: {'C': 0.5}
Best KNeighbors parameters: {'weights': 'distance', 'p': 2, 'n_neighbors': 35}
Voting Classifier accuracy: 0.8338386901152214
Voting Classifier precision: 0.5030674846625767
Voting Classifier recall: 0.5942028985507246
Voting Classifier F1: 0.5448504983388704


## Evaluación del modelo

In [22]:
# evaluate model
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score

accuracy = accuracy_score(Y_test, y_pred)
f1 = f1_score(Y_test, y_pred)
recall_score = recall_score(Y_test, y_pred)
precision_score = precision_score(Y_test, y_pred)
auc = roc_auc_score(Y_test, y_pred)

print(f'Accuracy: {accuracy}')
print(f'F1 score: {f1}')
print(f'Recall: {recall_score}')
print(f'Precision: {precision_score}')
print(f'AUC: {auc}')

Accuracy: 0.8502122498483929
F1 score: 0.5400372439478585
Recall: 0.5253623188405797
Precision: 0.5555555555555556
AUC: 0.7204378964923946


## Creacion de sometimiento

In [20]:
# use the model to predict the test data
y_test_pred = model.predict(data_test_clean)

# save the predictions to a csv file
submission = pd.DataFrame({'id': ids, 'resultado': y_test_pred})
submission.to_csv('submission.csv', index=False)

Grupo:
- Esteban Salazar Arbeláez
- Juan Francisco Ramírez Escobar
- Javier Alejandro Moyano Cipamocha